In [0]:
%sql

SELECT Code_text, COUNT(DISTINCT PERSON_ID) FROM 4_prod.rde.rde_all_diagnosis
WHERE Code_text ILIKE '%pneumothorax%' OR Code_text ILIKE '%haemothorax%'
GROUP BY Code_text


In [0]:
%sql
SELECT COUNT(DISTINCT PERSON_ID) FROM 4_prod.rde.rde_all_diagnosis
WHERE Code_text ILIKE '%pneumothorax%' OR Code_text ILIKE '%haemothorax%'


In [0]:
%sql
SELECT * FROM 4_prod.pacs_dlt.pacs_examcode_dict
WHERE modality_id = 77477000 -- CT
AND sct_fsn ILIKE '%chest%'
LIMIT 100

In [0]:
%sql

WITH pid AS (
SELECT DISTINCT PERSON_ID FROM 4_prod.rde.rde_all_diagnosis
WHERE Code_text ILIKE '%pneumothorax%' OR Code_text ILIKE '%haemothorax%'
)
SELECT COUNT(DISTINCT ACCESSIONNBR) FROM 4_prod.pacs.imaging_metadata AS im
INNER JOIN pid ON pid.PERSON_ID = im.PERSONID
WHERE ExamCode = 'CCHES'



In [0]:
%sql

WITH pid AS (
SELECT DISTINCT PERSON_ID FROM 4_prod.rde.rde_all_diagnosis
WHERE Code_text ILIKE '%pneumothorax%' OR Code_text ILIKE '%haemothorax%'
)
SELECT * FROM 4_prod.pacs.imaging_metadata AS im
INNER JOIN pid ON pid.PERSON_ID = im.PERSONID
WHERE ExamCode = 'CCHES' AND (RequestQuestion ILIKE '%pneumothorax%' OR RequestQuestion ILIKE '%haemothorax%')
LIMIT 1000

In [0]:
%sql

WITH pid AS (
SELECT DISTINCT PERSON_ID FROM 4_prod.rde.rde_all_diagnosis
WHERE Code_text ILIKE '%pneumothorax%' OR Code_text ILIKE '%haemothorax%'
)
SELECT * FROM 4_prod.pacs.imaging_metadata AS im
INNER JOIN pid ON pid.PERSON_ID = im.PERSONID
WHERE ExamCode = 'CCHES' AND (RequestQuestion ILIKE '%pneumothorax%' OR RequestQuestion ILIKE '%haemothorax%')
LIMIT 1000

In [0]:
%sql

WITH pid AS (
  SELECT DISTINCT PERSON_ID FROM 4_prod.rde.rde_all_diagnosis
  WHERE Code_text ILIKE '%pneumothorax%' OR Code_text ILIKE '%haemothorax%'
),
an AS (
  SELECT DISTINCT accessionnbr FROM 4_prod.pacs.imaging_metadata AS im
  INNER JOIN pid ON pid.PERSON_ID = im.PERSONID
  WHERE ExamCode = 'CCHES'
),
ir AS (
  SELECT personid, reporteventid, r.accessionnbr FROM 4_prod.pacs.imaging_report AS r
  INNER JOIN an ON an.accessionnbr = r.accessionnbr
  WHERE r.ExamCode = 'CCHES'
)
SELECT COUNT(DISTINCT accessionnbr) FROM ir
INNER JOIN 4_prod.rde.rde_blobdataset AS b ON ir.reporteventid = b.eventid
WHERE BlobContents ILIKE '%pneumothorax%'
LIMIT 100


In [0]:
%sql

WITH pid AS (
  SELECT DISTINCT PERSON_ID FROM 4_prod.rde.rde_all_diagnosis
  WHERE Code_text ILIKE '%pneumothorax%' OR Code_text ILIKE '%haemothorax%'
),
an AS (
  SELECT DISTINCT accessionnbr FROM 4_prod.pacs.imaging_metadata AS im
  INNER JOIN pid ON pid.PERSON_ID = im.PERSONID
  WHERE ExamCode = 'CCHES'
),
ir AS (
  SELECT personid, reporteventid, r.accessionnbr FROM 4_prod.pacs.imaging_report AS r
  INNER JOIN an ON an.accessionnbr = r.accessionnbr
  WHERE r.ExamCode = 'CCHES'
)
SELECT COUNT(DISTINCT accessionnbr) FROM ir
INNER JOIN 4_prod.rde.rde_blobdataset AS b ON ir.reporteventid = b.eventid
WHERE BlobContents ILIKE '%haemothorax%'
LIMIT 100

In [0]:
%sql

WITH pid AS (
  SELECT DISTINCT PERSON_ID FROM 4_prod.rde.rde_all_diagnosis
  WHERE Code_text ILIKE '%pneumothorax%' OR Code_text ILIKE '%haemothorax%'
),
an AS (
  SELECT DISTINCT accessionnbr FROM 4_prod.pacs.imaging_metadata AS im
  INNER JOIN pid ON pid.PERSON_ID = im.PERSONID
  WHERE ExamCode = 'CCHES'
),
ir AS (
  SELECT personid, reporteventid, r.accessionnbr FROM 4_prod.pacs.imaging_report AS r
  INNER JOIN an ON an.accessionnbr = r.accessionnbr
  WHERE r.ExamCode = 'CCHES'
)
SELECT COUNT(DISTINCT accessionnbr) FROM ir
INNER JOIN 4_prod.rde.rde_blobdataset AS b ON ir.reporteventid = b.eventid
WHERE BlobContents ILIKE '%pneumothorax%' OR BlobContents ILIKE '%haemothorax%'


In [0]:
import sys
sys.path.append("/Workspace/Shared/ADC-DB/Dev/pacs/utils")
import pacs_data_transformations as DT

In [0]:
q = "SELECT * FROM 5_projects.dar071.upload_20260629"
df = spark.sql(q)
display(df)

In [0]:
from pyspark.sql import functions as F
patterns = DT.createMillRefRegexPatternList()
df = df.withColumn("parsedAccessionNbr", DT.millRefToAccessionNbr(patterns, F.col("reference_nbr")))
display(df.select("parsedAccessionNbr").distinct())

In [0]:
df = spark.read.csv("/Volumes/1_inland/sectra/vone/dar071_20260724/extracted_dcm_tags.csv", header=True, inferSchema=True)
display(df)

In [0]:
dar_071_df = spark.table("5_projects.dar071.dar_071").withColumnRenamed("AccessionNumber", "requested_accession_number")
joined_df = df.join(dar_071_df, df.accession_number == dar_071_df.requested_accession_number, "right")
display(joined_df.filter("accession_number IS NULL"))

In [0]:
joined_df.write.mode("overwrite").option("header", True).csv("/Volumes/1_inland/sectra/vone/dar071_20260724/extracted_dcm_tags.csv")